In [1]:
import pandas as pd
from collections import Counter



# Tokenización
Se usará BPE (Byte procesing Encoding)

In [2]:
# convertir palabras a minusculas, eliminar acentos y caracteres especiales
def normalize(word):
    word = word.lower()
    word = word.replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u")
    word = "".join(c for c in word if c.isalnum())
    return word

In [3]:

NUMBER_OF_MERGES = 300

In [4]:
dinos = pd.read_csv("../000 data/Dinosours.csv")["nombre"]
dinos = dinos.apply(normalize)


In [5]:

def get_pairs(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_pair(pair, vocab):
    new_vocab = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word, freq in vocab.items():
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = freq
    return new_vocab

vocab = {" ".join(list(word)) + " </EOW>": 1 for word in dinos}

merge_rules = []

for i in range(NUMBER_OF_MERGES):
    pairs = get_pairs(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    vocab = merge_pair(best, vocab)
    merge_rules.append({"paso": i+1, "merge": f"{best[0]} + {best[1]}", "token_resultante": "".join(best), "frecuencia": pairs[best]})


In [6]:

# Tokens finales por dinosaurio
rows = []
for dino in dinos:
    tokens = " ".join(list(dino)) + " </EOW>"
    for a, b in [r["merge"].split(" + ") for r in merge_rules]:
        tokens = tokens.replace(f"{a} {b}", f"{a}{b}")
    rows.append({"dinosaurio": dino, "tokens": tokens.strip()})

df_tokens = pd.DataFrame(rows)
df_merges = pd.DataFrame(merge_rules)


In [7]:

import os
os.makedirs("../001 Proprocesamiento/tokens", exist_ok=True)

df_tokens.to_csv("../001 Proprocesamiento/tokens/dinos_tokens.csv", index=False)
df_merges.to_csv("../001 Proprocesamiento/tokens/merges.csv", index=False)

unique_tokens = df_tokens["tokens"].str.split(expand=True).stack().unique()

print("Cantidad de tokens únicos:", len(unique_tokens))

unique_tokens_df = pd.DataFrame(unique_tokens, columns=["token"])
unique_tokens_df.to_csv("../001 Proprocesamiento/tokens/unique_tokens.csv", index=True)

Cantidad de tokens únicos: 1457
